In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import seaborn as sns

# ------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------
df = sns.load_dataset("iris")

# Encode species into 0,1,2
encoder = LabelEncoder()
df['species'] = encoder.fit_transform(df['species'])

# Use only 2 features to allow visualization later
df = df[['sepal_length', 'petal_length', 'species']]

X = df.iloc[:, 0:2].values
y = df.iloc[:, -1].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# ------------------------------------------------------
# 2. Helper Functions
# ------------------------------------------------------

def softmax(z):
    exp_z = np.exp(z - np.max(z))      # stability improvement
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def one_hot(y, num_classes):
    oh = np.zeros((len(y), num_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

# Convert y_train to one-hot
y_train_oh = one_hot(y_train, 3)

# ------------------------------------------------------
# 3. Initialize Weights
# ------------------------------------------------------
n_features = X_train.shape[1]    # 2 features
n_classes = 3

W = np.zeros((n_features, n_classes))   # weights
b = np.zeros((1, n_classes))            # bias

lr = 0.05     # learning rate
epochs = 5000

# ------------------------------------------------------
# 4. TRAINING LOOP (Gradient Descent)
# ------------------------------------------------------
for i in range(epochs):

    # Forward pass
    z = np.dot(X_train, W) + b         # shape: (N, 3)
    y_hat = softmax(z)                 # predicted probabilities

    # Loss (Cross Entropy)
    loss = -np.mean(np.sum(y_train_oh * np.log(y_hat + 1e-9), axis=1))

    # Gradients
    dz = y_hat - y_train_oh                   # (N,3)
    dW = np.dot(X_train.T, dz) / len(X_train) # (2,3)
    db = np.mean(dz, axis=0, keepdims=True)   # (1,3)

    # Update rule
    W -= lr * dW
    b -= lr * db

    if i % 500 == 0:
        print("Epoch:", i, " Loss:", loss)

# ------------------------------------------------------
# 5. PREDICTION
# ------------------------------------------------------
def predict(X):
    z = np.dot(X, W) + b
    probs = softmax(z)
    return np.argmax(probs, axis=1)

y_pred = predict(X_test)

# ------------------------------------------------------
# 6. Accuracy
# ------------------------------------------------------
accuracy = np.mean(y_pred == y_test)
print("Accuracy =", accuracy)

